# Dynamic Task Decomposition with Parallel Sub-Agent Execution 🧩

A hierarchical LangGraph system that decomposes complex tasks into subtasks, dynamically spawns parallel sub-agents using the Send() API, aggregates results with state reducers, validates completeness, and synthesizes final output.

## Executive Summary

### What This Agent Does

A sophisticated LangGraph-based system that:
1. Takes a complex user task (research report, analysis, content creation)
2. Automatically breaks it into parallelizable subtasks
3. Dynamically spawns parallel sub-agents using LangGraph's Send() API
4. Aggregates results reliably with state reducers
5. Validates completeness and quality
6. Re-decomposes weak or incomplete sections if needed
7. Synthesizes a comprehensive final output

### Why This Solves a Real Pain Point

- **Problem**: Complex tasks overwhelm single agents, leading to shallow results
- **Impact**: Users get generic answers instead of comprehensive analysis
- **Our Solution**: Automated parallel execution with dynamic sub-agent spawning

In [ ]:
from dotenv import load_dotenv
import os, json, time, re, sys
from typing import TypedDict, List, Annotated, Literal, Optional
import operator as op
from pathlib import Path

env_candidates = [
    Path(".env"),
    Path("../.env"),
    Path.cwd() / "../.env",
]
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)

# --- Load Groq ---
if not os.getenv("GROQ_API_KEY"):
    for candidate in env_candidates:
        if candidate and candidate.exists():
            with open(candidate) as f:
                for line in f:
                    if line.startswith("GROQ_API_KEY"):
                        key = line.split("=", 1)[1].strip().strip("\"'")
                        os.environ["GROQ_API_KEY"] = key
                        break

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")

# --- Load Google Gemini ---
if not os.getenv("GOOGLE_API_KEY"):
    for candidate in env_candidates:
        if candidate and candidate.exists():
            with open(candidate) as f:
                for line in f:
                    if line.startswith("GOOGLE_API_KEY"):
                        key = line.split("=", 1)[1].strip().strip("\"'")
                        os.environ["GOOGLE_API_KEY"] = key
                        break

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
GOOGLE_MODEL = os.getenv("GOOGLE_MODEL", "gemini-2.0-flash")

print(f"Groq:  model={GROQ_MODEL}, key={'SET' if GROQ_API_KEY else 'NOT SET'}")
print(f"Gemini: model={GOOGLE_MODEL}, key={'SET' if GOOGLE_API_KEY else 'NOT SET'}")

In [ ]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from groq import RateLimitError

def get_llm(temperature: float = 0.3):
    if GROQ_API_KEY:
        return ChatGroq(
            model=GROQ_MODEL,
            temperature=temperature,
            api_key=GROQ_API_KEY
        )
    if GOOGLE_API_KEY:
        return ChatGoogleGenerativeAI(
            model=GOOGLE_MODEL,
            temperature=temperature,
            api_key=GOOGLE_API_KEY
        )
    raise RuntimeError("No API key configured for either Groq or Gemini")

def get_fallback_llm(temperature: float = 0.3):
    if GROQ_API_KEY and GOOGLE_API_KEY:
        return ChatGoogleGenerativeAI(
            model=GOOGLE_MODEL,
            temperature=temperature,
            api_key=GOOGLE_API_KEY
        )
    return None

def llm_invoke(llm: BaseChatModel, prompt: str, max_retries: int = 2) -> str:
    fallback_llm = get_fallback_llm()
    for attempt in range(max_retries):
        try:
            return llm.invoke(prompt).content
        except Exception as e:
            if fallback_llm:
                print(f"  {type(e).__name__} on primary, switching to Gemini...")
                for fb_attempt in range(3):
                    try:
                        return fallback_llm.invoke(prompt).content
                    except Exception as fb_e:
                        wait = min(2 ** fb_attempt, 10)
                        print(f"  Gemini also {type(fb_e).__name__}, retry in {wait}s...")
                        time.sleep(wait)
                raise RuntimeError("Both providers failed for this call")
            wait = min(2 ** attempt * 2, 20)
            print(f"  {type(e).__name__}, retrying in {wait}s (attempt {attempt+1}/{max_retries})...")
            time.sleep(wait)
    raise RuntimeError(f"Failed after {max_retries} retries")

print("All imports successful")

## 1. State Schema

Define the state structure that flows through the graph. The `parallel_results` field uses `operator.add` as a reducer so results from parallel workers accumulate correctly.

In [ ]:
class TaskDecompositionState(TypedDict):
    """State schema for the task decomposition and parallel execution system."""
    user_task: str
    subtasks: List[dict]
    parallel_results: Annotated[List[str], op.add]
    num_subtasks: int
    max_iterations: int
    decomposition_iterations: int
    completion_rate: float
    avg_quality_score: float
    weak_sections: List[str]
    failed_subtasks: List[dict]
    synthesized_output: str
    final_quality_score: float
    quality_assessment: str

print("State schema defined")

## 2. Decomposer Agent

Breaks down the user's complex task into well-defined, parallelizable subtasks. Each subtask includes a unique ID, description, assigned agent type, and payload.

In [ ]:
def decomposer_agent(state: TaskDecompositionState) -> TaskDecompositionState:
    prompt = f"""
    You are a task decomposition expert. Break this task into parallelizable subtasks:

    Task: "{state["user_task"]}"

    Guidelines:
    1. Each subtask must be independently executable
    2. Identify 3-6 subtasks covering different aspects
    3. Assign each to an agent type: research_agent, analysis_agent, or writing_agent
    4. Include a unique subtask_id
    5. Set status to 'pending'

    Return ONLY valid JSON (no markdown):
    {{
        "subtasks": [
            {{
                "subtask_id": "st_1",
                "description": "...",
                "assigned_agent": "research_agent",
                "payload": {{}},
                "status": "pending"
            }}
        ]
    }}
    """
    llm = get_llm(temperature=0.5)
    content = llm_invoke(llm, prompt)
    if "```json" in content:
        content = content.split("```json")[1].split("```")[0].strip()
    elif "```" in content:
        content = content.split("```")[1].split("```")[0].strip()
    try:
        parsed = json.loads(content)
        subtasks = parsed.get("subtasks", [])
    except json.JSONDecodeError:
        print("Failed to parse decomposer output, using fallback")
        subtasks = [
            {"subtask_id": "st_1", "description": f"Research {state["user_task"][:50]}",
             "assigned_agent": "research_agent", "payload": {}, "status": "pending"}
        ]
    print(f"Decomposed into {len(subtasks)} subtask(s)")
    return {
        "subtasks": subtasks,
        "num_subtasks": len(subtasks)
    }

print("Decomposer defined")

## 3. Worker Router (Send() API)

This is the key function demonstrating LangGraph's advanced `Send()` API for dynamic parallel execution. It creates one `Send()` object per subtask, each carrying its own state payload so workers run independently and concurrently.

In [ ]:
INSTANCE_COUNTER = 0

def worker_router(state: TaskDecompositionState) -> List[Send]:
    """Conditional edge: spawns parallel workers via the Send() API."""
    global INSTANCE_COUNTER
    subtasks = state.get("subtasks", [])
    pending = [s for s in subtasks if s.get("status") == "pending"]
    sends = []
    for s in pending:
        INSTANCE_COUNTER += 1
        sends.append(Send(
            "parallel_worker",
            {"subtask": s, "user_task": state["user_task"], "instance_id": INSTANCE_COUNTER}
        ))
    remaining = len(subtasks) - len(pending)
    print(f"  Spawning {len(sends)} parallel worker(s) ({remaining} remaining for next round)...")
    for sd in sends[:3]:
        p = sd.payload["subtask"]
        print(f"    Worker [{sd.payload['instance_id']}]: {p['assigned_agent']} - {p['description'][:60]}...")
    return sends

print("Worker router defined")

## 4. Parallel Worker

Executes an individual subtask. Receives partial state from the Send() payload. Each worker runs its own LLM call based on the assigned agent type, evaluates output quality, and returns its result.

In [ ]:
def parallel_worker(state: dict) -> dict:
    """Executes an individual subtask. Receives partial state from Send() payload."""
    subtask = state["subtask"]
    user_task = state["user_task"]
    instance_id = state.get("instance_id", 0)

    agent_type = subtask.get("assigned_agent", "research_agent")
    prompt = f"""
    You are a {agent_type.replace('_', ' ')}. Perform this subtask:

    Subtask: {subtask['description']}
    Context: Part of larger task: "{user_task}"

    Provide comprehensive, well-structured output.
    Return ONLY valid JSON:
    {{
        "subtask_id": "{subtask['subtask_id']}",
        "output": "your detailed output here",
        "quality_score": 7.5
    }}
    """
    llm = get_llm(temperature=0.7)
    result = llm_invoke(llm, prompt)
    if "```json" in result:
        result = result.split("```json")[1].split("```")[0].strip()
    elif "```" in result:
        result = result.split("```")[1].split("```")[0].strip()
    try:
        parsed = json.loads(result)
        quality = parsed.get("quality_score", 7.0)
    except json.JSONDecodeError:
        parsed = {"subtask_id": subtask["subtask_id"], "output": result, "quality_score": 7.0}
        quality = 7.0
    print(f"  Worker [{instance_id}] done (quality: {quality}/10)")
    return {"parallel_results": [json.dumps(parsed)]}

print("Parallel worker defined")

## 5. Completeness Validator

Maps completed results back to their subtasks, computes completion rate and average quality, and identifies weak sections that need re-decomposition.

In [ ]:
def completeness_validator(state: TaskDecompositionState) -> TaskDecompositionState:
    subtasks = state.get("subtasks", [])
    results = state.get("parallel_results", [])
    for r in results:
        try:
            data = json.loads(r) if isinstance(r, str) else r
            sid = data.get("subtask_id")
            for s in subtasks:
                if s.get("subtask_id") == sid:
                    s["status"] = "completed"
                    s["result"] = data.get("output", "")
                    s["quality_score"] = data.get("quality_score", 7.0)
                    break
        except Exception:
            pass
    for s in subtasks:
        if s.get("status") in ("pending", None) or s.get("result") is None:
            s["status"] = "failed"
    completed = [s for s in subtasks if s["status"] == "completed"]
    failed = [s for s in subtasks if s["status"] == "failed"]
    completion_rate = len(completed) / len(subtasks) if subtasks else 0
    quality_scores = [s["quality_score"] for s in completed if s.get("quality_score") is not None]
    avg_quality = sum(quality_scores) / len(quality_scores) if quality_scores else 0
    weak_sections = [s["description"] for s in completed if s.get("quality_score") is not None and s["quality_score"] < 6.0]
    print(f"Completion: {len(completed)}/{len(subtasks)} subtasks done")
    print(f"Avg quality: {avg_quality:.1f}/10")
    print(f"Weak sections: {len(weak_sections)}")
    return {
        "subtasks": subtasks,
        "completion_rate": completion_rate,
        "avg_quality_score": avg_quality,
        "weak_sections": weak_sections,
        "failed_subtasks": failed
    }

print("Completeness validator defined")

## 6. Re-Decomposer Agent

Splits failed or low-quality subtasks into finer-grained subtasks. Includes a hard ceiling (MAX_TOTAL_SUBTASKS=15) to prevent exponential blowup.

In [ ]:
MAX_TOTAL_SUBTASKS = 15

def re_decomposer_agent(state: TaskDecompositionState) -> TaskDecompositionState:
    subtasks = state.get("subtasks", [])
    user_task = state.get("user_task", "")
    failed_or_weak = [
        s for s in subtasks
        if s.get("status") in ("failed", "pending")
        or (s.get("quality_score") is not None and s["quality_score"] < 5.5)
    ]
    good_subtasks = [s for s in subtasks if s not in failed_or_weak]
    slots_available = MAX_TOTAL_SUBTASKS - len(good_subtasks)
    if slots_available <= 0:
        print(f"  Subtask cap ({MAX_TOTAL_SUBTASKS}) reached. Skipping re-decompose.")
        return {"subtasks": good_subtasks,
                "decomposition_iterations": state.get("decomposition_iterations", 1) + 1}
    to_redecompose = failed_or_weak[:slots_available]
    print(f"Re-decomposing {len(to_redecompose)} weak/failed subtask(s)...")
    new_subtasks = []
    for weak_subtask in to_redecompose:
        available = slots_available - len(new_subtasks)
        if available <= 0:
            break
        children = min(2, available)
        prompt = f"""Break this subtask into exactly {children} finer-grained subtasks:

Original subtask: {weak_subtask['description']}
Context: Part of larger task: "{user_task}"

Return ONLY valid JSON (no markdown):
{{
    "new_subtasks": [
        {{
            "subtask_id": "new_1",
            "description": "...",
            "assigned_agent": "research_agent",
            "payload": {{}}
        }}
    ]
}}"""
        llm = get_llm(temperature=0.5)
        content = llm_invoke(llm, prompt)
        if "```json" in content:
            content = content.split("```json")[1].split("```")[0].strip()
        elif "```" in content:
            content = content.split("```")[1].split("```")[0].strip()
        try:
            new_breakdown = json.loads(content)
            for ns in new_breakdown.get("new_subtasks", [])[:children]:
                ns["status"] = "pending"
                ns["result"] = None
                ns["quality_score"] = None
                new_subtasks.append(ns)
        except json.JSONDecodeError:
            print(f"  Failed to re-decompose: {weak_subtask['subtask_id']}")
    updated_subtasks = good_subtasks + new_subtasks
    print(f"Replaced {len(to_redecompose)} weak tasks with {len(new_subtasks)} new ones. Total: {len(updated_subtasks)}")
    return {
        "subtasks": updated_subtasks,
        "decomposition_iterations": state.get("decomposition_iterations", 1) + 1,
    }

print("Re-decomposer agent defined")

## 7. Synthesis Agent

Combines all completed subtask results into a comprehensive, coherent final output.

In [ ]:
def synthesis_agent(state: TaskDecompositionState) -> TaskDecompositionState:
    completed = [s for s in state.get("subtasks", []) if s.get("status") == "completed"]
    all_results = [s.get("result", "") for s in completed if s.get("result")]
    if not all_results:
        return {
            "synthesized_output": "No completed subtasks to synthesize.",
            "num_subtasks": state.get("num_subtasks", 0)
        }
    prompt = f"""
    You are a synthesis expert. Combine these subtask results into a comprehensive, coherent final output:

    Original task: "{state['user_task']}"

    Subtask results:
    {json.dumps(all_results, indent=2)[:8000]}

    Your task:
    1. Organize results logically (introduction, body sections, conclusion)
    2. Add transitions and connections between sections
    3. Ensure consistent tone and style
    4. Eliminate redundancy
    5. Add overarching insights that emerge from combining all results
    6. Create a compelling narrative flow

    Return the synthesized final output.
    """
    llm = get_llm(temperature=0.4)
    output = llm_invoke(llm, prompt)
    print("Synthesis complete.")
    return {
        "synthesized_output": output
    }

print("Synthesis agent defined")

## 8. Quality Checker

Evaluates the final synthesized output for completeness, accuracy, clarity, coherence, and depth.

In [ ]:
def quality_checker(state: TaskDecompositionState) -> TaskDecompositionState:
    output = state.get("synthesized_output", "")
    if not output:
        return {
            "final_quality_score": 0.0,
            "quality_assessment": "No output to evaluate."
        }
    prompt = f"""
    Evaluate this final output for quality.
    Task: "{state['user_task']}"
    Output: {output[:3000]}
    Score each 0-10: completeness, accuracy, clarity, coherence, depth.
    Return ONLY raw JSON with final_quality_score (avg float), quality_assessment (str), sections_to_improve (list).
    No markdown, no code fences.
    Example: {{"final_quality_score": 8.5, "quality_assessment": "Good report covering all areas.", "sections_to_improve": []}}
    """
    content = llm_invoke(get_llm(temperature=0.3), prompt).strip()
    if content.startswith("```"):
        content = content.split("\n", 1)[-1]
        if "```" in content:
            content = content.rsplit("```", 1)[0].strip()
    try:
        assessment = json.loads(content)
    except json.JSONDecodeError:
        import re as _re
        match = _re.search(r'\{[^{}]*"final_quality_score"[^{}]*\}', content, _re.DOTALL)
        if match:
            try:
                assessment = json.loads(match.group())
            except json.JSONDecodeError:
                assessment = {"final_quality_score": 7.0, "quality_assessment": "Parse failed.", "sections_to_improve": []}
        else:
            assessment = {"final_quality_score": 7.0, "quality_assessment": "Parse failed.", "sections_to_improve": []}
    print(f"Final quality score: {assessment.get('final_quality_score', 0):.1f}/10")
    return assessment

print("Quality checker defined")

## 9. Gate Functions

Conditional edge functions that determine graph routing based on state.

In [ ]:
def quality_gate(state: TaskDecompositionState) -> str:
    completion_rate = state.get("completion_rate", 0)
    avg_quality = state.get("avg_quality_score", 0)
    weak_sections = state.get("weak_sections", [])
    iterations = state.get("decomposition_iterations", 1)
    max_iter = state.get("max_iterations", 3)
    total_subtasks = len(state.get("subtasks", []))
    if iterations >= max_iter:
        print(f"Max iterations ({max_iter}) reached. Proceeding to synthesis.")
        return "synthesize"
    if total_subtasks >= MAX_TOTAL_SUBTASKS:
        print(f"Subtask cap ({MAX_TOTAL_SUBTASKS}) reached. Proceeding to synthesis.")
        return "synthesize"
    if completion_rate < 0.8:
        print(f"Completion rate {completion_rate:.0%} < 80%. Re-decomposing...")
        return "re_decompose"
    if avg_quality < 5.5:
        print(f"Avg quality {avg_quality:.1f} < 5.5. Re-decomposing...")
        return "re_decompose"
    if len(weak_sections) > 0:
        print(f"{len(weak_sections)} weak section(s) found. Re-decomposing...")
        return "re_decompose"
    print("All quality checks passed. Proceeding to synthesis.")
    return "synthesize"

print("Gate functions defined")

## 10. Building the Graph

Wire all nodes and edges together into a LangGraph StateGraph with dynamic parallel execution via the Send() API.

In [ ]:
def build_task_decomposition_graph():
    graph = StateGraph(TaskDecompositionState)
    graph.add_node("decomposer", decomposer_agent)
    graph.add_node("parallel_worker", parallel_worker)
    graph.add_node("validator", completeness_validator)
    graph.add_node("re_decomposer", re_decomposer_agent)
    graph.add_node("synthesis", synthesis_agent)
    graph.add_node("quality_checker", quality_checker)
    graph.add_edge(START, "decomposer")
    graph.add_conditional_edges(
        "decomposer", worker_router, ["parallel_worker"]
    )
    graph.add_edge("parallel_worker", "validator")
    graph.add_conditional_edges(
        "validator", quality_gate,
        {"synthesize": "synthesis", "re_decompose": "re_decomposer"}
    )
    graph.add_conditional_edges(
        "re_decomposer", worker_router, ["parallel_worker"]
    )
    graph.add_edge("synthesis", "quality_checker")
    graph.add_edge("quality_checker", END)
    return graph.compile()

app = build_task_decomposition_graph()
print("Graph compiled successfully")

## 11. Example Run

Let's test the parallel task decomposition agent with a comprehensive research report task.

In [ ]:
user_task = "Write a comprehensive report on AI in healthcare in 2026, covering diagnostic tools, patient data privacy, regulatory landscape, ethical considerations, and future trends."

initial_state = {
    "user_task": user_task,
    "subtasks": [],
    "parallel_results": [],
    "num_subtasks": 0,
    "max_iterations": 3,
    "decomposition_iterations": 1,
    "completion_rate": 0.0,
    "avg_quality_score": 0.0,
    "weak_sections": [],
    "failed_subtasks": [],
    "synthesized_output": "",
    "final_quality_score": 0.0,
    "quality_assessment": ""
}

print("Starting parallel task decomposition and execution...")
print(f"Task: {user_task[:80]}...")
start_time = time.time()
result = app.invoke(initial_state)
elapsed = time.time() - start_time
print(f"\nTotal time: {elapsed:.1f}s")
print(f"Final quality score: {result.get('final_quality_score', 'N/A')}/10")
print(f"\n=== SYNTHESIZED OUTPUT ===\n{result.get('synthesized_output', 'No output')[:1500]}...\n")

## Key Takeaways

1. **Dynamic Parallel Execution**: The Send() API enables spawning parallel workers dynamically at runtime
2. **Quality Gating**: Automatic re-decomposition of weak sections ensures high-quality output
3. **Scalable Architecture**: The same pattern can handle research, content creation, analysis, and more
4. **Production Feature**: State reducers, error handling, and quality thresholds make this production-ready

### When to Use This Pattern

- Complex research or analysis tasks with multiple distinct facets
- Content creation requiring diverse expertise (research, analysis, writing)
- Any task that benefits from divide-and-conquer with parallel execution
- Scenarios where quality gates and iterative refinement add value